In [38]:
from src.schemas.evaluation import EvaluationSummaryRow
import json
from pathlib import Path

evaluation_summary_path = Path("./data/results/evaluation_results_full_correct.jsonl")

evaluations = []

for line in evaluation_summary_path.open():
    try:
        evaluations.append(json.loads(line))
    except json.JSONDecodeError:
        print(f"Failed to decode line: {line}")


In [39]:
len(evaluations)

110

In [35]:
for line in evaluations:
    line["model_id"] = line["model"]
    del line["model"]
    line["judge_response"] = line["parsed_response"]
    del line["parsed_response"]
    raw_response = line["raw_response"]
    line["predicted_level"] = line["predicted_level"].lower()
    line["predicted_level"] = None if not line["predicted_level"] else line["predicted_level"]
    if line["judge_response"] is not None:
        judge_response = {
            "level" : line["judge_response"]["level"].lower(),
            "analysis" : line["judge_response"]["explanation"],
        }
        line["judge_response"] = {
            "analysis" : judge_response,
            "raw_response" : raw_response,
            "thoughts" : None
        }
    else:
        line["judge_response"] = {
            "analysis" : None,
            "raw_response" : raw_response,
            "thoughts" : None
        }
    

In [32]:
EvaluationSummaryRow(**evaluations[0])

EvaluationSummaryRow(model_id='gpt-oss:20b', kind='ollama', sample_index=0, expected_level='functional_error', predicted_level='correct', correct=False, judge_response=JudgeResponse(analysis=JudgeAnalysis(level='correct', analysis='The provided code is syntactically correct and implements the RGB to HSV conversion algorithm accurately. All variables are defined, divisions are safe (division by zero is avoided by checking `v == 0` before computing saturation), and the hue calculation follows the standard formula. No runtime or functional errors are present.'), raw_response='```json\n{\n    "level": "correct",\n    "explanation": "The provided code is syntactically correct and implements the RGB to HSV conversion algorithm accurately. All variables are defined, divisions are safe (division by zero is avoided by checking `v == 0` before computing saturation), and the hue calculation follows the standard formula. No runtime or functional errors are present."\n}\n```', thoughts=None))

In [36]:
evaluation_summary = [EvaluationSummaryRow(**evaluation) for evaluation in evaluations]

In [37]:
with evaluation_summary_path.open("w") as f:
    for evaluation in evaluation_summary:
        json.dump(evaluation.model_dump(), f)
        f.write("\n")